# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
[https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access and print the dataset metadata (name and description)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"Published: {dataset.metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets and their @id
record_sets = dataset.record_sets

print("Record Sets:")
for rs in record_sets:
    print(f"- {rs['@id']} (name: {rs.get('name', '[unknown]')})")
    if 'fields' in rs:
        print("  Fields:")
        for field in rs['fields']:
            print(f"    - {field['@id']} (name: {field.get('name', '[unknown]')}, type: {field.get('dataType', '[unknown]')})")

# For illustration, print the first few records from the main record set
if record_sets:
    main_record_set_id = record_sets[0]['@id']
    for x in dataset.records(record_set=main_record_set_id):
        print(x)
        break  # Print only the first record for overview

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Collect all record set @id's for extraction
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print("Record Set IDs:", record_set_ids)

dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nColumns in record set {record_set_id}:")
    print(df.columns.tolist())

# Show the first few rows from the main record set
main_record_set_id = record_set_ids[0]
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In [ ]:
# Choose a numeric field for analysis from the main record set
# Find numeric fields IDs (e.g., GAD-7 score, PHQ-9 score, PSQ score)
main_rs = dataset.record_sets[0]
numeric_fields = [f for f in main_rs['fields'] if f.get('dataType') in ['schema:Integer', 'schema:Float', 'Integer', 'Float']]
print("Numeric fields in record set:")
for nf in numeric_fields:
    print(f"- {nf['@id']} (name: {nf.get('name')})")

# Example: Assume GAD-7 score field @id is 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/gad7_score'
# Substitute with actual @id found above
if numeric_fields:
    numeric_field_id = numeric_fields[0]['@id']
    numeric_field_name = numeric_fields[0]['name'] if 'name' in numeric_fields[0] else numeric_field_id

    df = dataframes[main_record_set_id]

    # Set a threshold for filtering (example: 10)
    threshold = 10
    if numeric_field_id in df.columns:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the field
        col = numeric_field_id
        filtered_df[col + '_normalized'] = (filtered_df[col] - filtered_df[col].mean()) / filtered_df[col].std()
        print(f"Normalized {col} for filtered records:")
        print(filtered_df[[col, col + '_normalized']].head())

        # Choose a group field (e.g., education or gender), get its @id
        group_fields = [f for f in main_rs['fields'] if f.get('dataType') == 'schema:Text']
        if group_fields:
            group_field_id = group_fields[0]['@id']
            if group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id)[col].mean().reset_index()
                print(f"Grouped mean of {col} by {group_field_id}:")
                print(grouped_df.head())
    else:
        print(f"Numeric field {numeric_field_id} not found in dataframe columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the numeric field (e.g. GAD-7 score)
if numeric_fields:
    col = numeric_field_id
    df = dataframes[main_record_set_id]
    if col in df.columns:
        plt.figure(figsize=(8, 5))
        sns.histplot(df[col].dropna(), bins=10, kde=True)
        plt.title(f"Distribution of {col}")
        plt.xlabel(col)
        plt.ylabel("Frequency")
        plt.show()

# If grouped_df was created, show a barplot
if 'grouped_df' in locals():
    plt.figure(figsize=(8, 5))
    sns.barplot(data=grouped_df, x=group_field_id, y=col)
    plt.title(f"Mean {col} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {col}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrated loading and exploration of the FAIR² mental health survey dataset from Kilifi County, Kenya using `mlcroissant`. We examined record sets and fields by their `@id`s, filtered and normalized survey scores, and visualized score distributions by group. The dataset structure and unique `@id` referencing enable robust, reproducible data analysis. For further investigation, additional fields and record sets can be explored following this template.